# Õppetund 11 - Model Context Protocol (MCP)

**Model Context Protocol (MCP)** on avatud standard, mis võimaldab agentidel käituse ajal dünaamiliselt tööriistu, ressursse ja andmeallikaid avastada ja kasutada. Selle asemel, et tööriistu agendi koodi kõvakodeerida, võimaldab MCP agentidel ühendada välistele serveritele, mis pakuvad võimekusi nõudmisel.

Selles õppetükis õpid:
- Mis on MCP ja miks see on oluline agendisüsteemide jaoks
- Kuidas MCP klient-serveri arhitektuur töötab
- Kuidas ehitada agente, mis kasutavad MCP-tüüpi tööriistade avastamist


## Seadistamine

**Eeltingimused:**
- Azure AI Foundry projekt koos juurutatud mudeliga
- Käivita `az login` autentimiseks


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on Model Context Protocol (MCP)?

MCP määratleb standardse viisi, kuidas tehisintellekti agendid saavad avastada ja suhelda väliste tööriistade ning andmeallikatega:

- **MCP Server**: Avaldab tööriistu, ressursse ja prompt'e standardse protokolli kaudu
- **MCP Client**: Agendi käitusaeg, mis ühendub serveritega ja avastab saadaolevaid võimekusi
- **Dynamic Discovery**: Agenditel ei ole vaja kõvakodeeritud tööriistu — nad avastavad, mis on käituse ajal saadaval

See on võimas lahendus laiendatavate agendisüsteemide ehitamiseks, kus uusi võimekusi saab lisada ilma agendi koodi muutmata.


## Kuidas MCP töötab

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP client) ühendub MCP serveriga
2. Server vastab saadaolevate tööriistade ja nende skeemide loendiga
3. Agent saab seejärel oma järeldamisprotsessi käigus kutsuda välja ükskõik millise avastatud tööriista
4. Tulemused liiguvad tagasi läbi sama protokolli


## MCP-tööriistade avastamise simuleerimine

Kuna päris MCP-server nõuab töötavat serveriprotsessi, demonstreerime mustrit, kasutades `@tool` funktsioone, mis simuleerivad seda, mida MCP-iga ühendatud majutusteenus pakuks.

Tootmises avastataks need tööriistad dünaamiliselt MCP-serverist, mitte ei määratletaks neid lokaalselt.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Agendi loomine MCP-laadsete tööriistadega


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP tootmiskeskkonnas

Tootmiskeskkonnas võimaldab MCP võimsaid mustreid:

- **Dünaamiline tööriistade avastamine**: Agendid ühenduvad MCP serveritega ja avastavad tööriistu käituse ajal
- **Eraldatud arhitektuur**: Tööriistapakkujad saavad uuendada sõltumatult agendist
- **Organisatsioonidevaheline jagamine**: Meeskonnad saavad avaldada võimekusi MCP serverite kaudu, mida iga agent saab kasutada
- **Microsoft Agent Frameworki tugi**: MAF-il on sisseehitatud MCP klienditugi `mcp` integratsiooni kaudu

Reaalse MCP serveri kasutamiseks MAF-iga ühenduksite läbi `hosted_mcp_tool()` või MCP kliendi integratsiooni.

**Lisateave:**
- [MCP spetsifikatsioon](https://modelcontextprotocol.io/)
- [Microsoft Agent Frameworki MCP tugi](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Kokkuvõte

Selles õppetükis õppisite:
- **MCP** on avatud standard agentide ja tööriistapakkujate vaheliseks dünaamiliseks tööriistade avastamiseks
- **klient-serveri arhitektuur** võimaldab agentidel käituse ajal võimeid avastada
- MCP võimaldab **laiendatavaid, lõdvalt seotud agentide süsteeme**, kus tööriistu saab lisada ilma koodimuudatusteta
- Microsoft Agent Framework pakub **sisseehitatud MCP tuge** tootmiskasutuseks


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Lahtiütlus:
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame tagada täpsust, tuleks arvestada, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokumenti selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste ega valede tõlgenduste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
